In [ ]:
import numpy as np
import qutip as qt

In [ ]:
def get_qubit_ham(N, wm=1.0, fixed_seed=False, indv_qubit=False, ds_dis=0.0, sigz_ham=False):
    if fixed_seed:
        np.random.seed(0)

    ds = np.random.uniform(-ds_dis, ds_dis, N)
    ds += 1.0

    sigz = qt.sigmaz()
    sigy = qt.sigmay()
    sigx = qt.sigmax()
    eye = qt.qeye(2)

    eyeList = [eye] * N

    qH0 = 0
    qH1 = 0
    qH1_list = []

    for i in range(N):
        ops0 = eyeList.copy()
        ops1 = eyeList.copy()

        if sigz_ham:
            ops0[i] = -0.5 * wm * sigz
            ops1[i] = ds[i] * sigx
        else:
            ops0[i] = -0.5 * wm * sigx
            ops1[i] = ds[i] * sigz

        qH0 += qt.tensor(ops0)

        if not indv_qubit:
            qH1 += qt.tensor(ops1)
        else:
            qH1_list.append(qt.tensor(ops1))

    eigenvalues, eigenstates = qH0.eigenstates()

    if not indv_qubit:
        return qH0, qH1, eigenvalues, eigenstates
    else:
        return qH0, qH1_list, eigenvalues, eigenstates
    
def get_dis_qubit_ham(qH0_dis, N, N_dis=None, ham_disorder=[0, 0, 0], fixed_seed=False):
    if N_dis == None:
        N_dis = N

    if fixed_seed:
        np.random.seed(0)

    if ham_disorder[0] != 0.0:
        zd = ham_disorder[0]
        hz = np.zeros(N)
        dis_sites = np.random.choice(N, size=N_dis, replace=False)
        hz[dis_sites] = np.random.uniform(-zd, zd, N_dis)

    if ham_disorder[1] != 0.0:
        yd = ham_disorder[1]
        hy = np.zeros(N)
        dis_sites = np.random.choice(N, size=N_dis, replace=False)
        hy[dis_sites] = np.random.uniform(-yd, yd, N_dis)

    if ham_disorder[2] != 0.0:
        xd = ham_disorder[2]
        hx = np.zeros(N)
        dis_sites = np.random.choice(N, size=N_dis, replace=False)
        hx[dis_sites] = np.random.uniform(-xd, xd, N_dis)

    sigz = qt.sigmaz()
    sigy = qt.sigmay()
    sigx = qt.sigmax()
    eye = qt.qeye(2)

    eyeList = [eye] * N

    ham_dis = qt.Qobj(np.zeros((2**N, 2**N)), dims=[[2]*N, [2]*N])

    for i in range(N):
        zops0 = eyeList.copy()
        yops0 = eyeList.copy()
        xops0 = eyeList.copy()

        if ham_disorder[0] != 0.0:
            zops0[i] = hz[i] * sigz

        if ham_disorder[1] != 0.0:
            yops0[i] = hy[i] * sigy

        if ham_disorder[2] != 0.0:
            xops0[i] = hx[i] * sigx
        
        ham_dis += qt.tensor(zops0)
        ham_dis += qt.tensor(yops0)
        ham_dis += qt.tensor(xops0)

    qH0_dis += ham_dis

    qeigenvalues, qeigenstates = qH0_dis.eigenstates()

    return qH0_dis, qeigenvalues, qeigenstates

In [ ]:
def giveMeScarVonNeumannEntrop(N, wd, tlist, disorder=[0, 0, 0], reals=1):
    scarEntangle = []
    for _ in range(reals):
        H0_clean, eigenvalues, eigenstates, psi0, basisList = get_scar_ham(N)
        H0, eigenvalues, eigenstates = get_dis_scar_ham(
            H0_clean,
            N,
            basisList,
            ham_disorder=disorder,
            fixed_seed=False
        )
        H1, driveWeights = get_scar_H1(N, basisList)

        args = {"A": 0.1, "omega": wd}
        H = qt.QobjEvo([H0, [H1, coeff]], args=args)
        psi_t = qt.sesolve(H, eigenstates[0], tlist)

        temp = []
        for state in psi_t.states:
            psi_full = embed_scar_state_to_full(state, basisList, N)
            rho_A = psi_full.ptrace(list(range(N//2)))
            temp.append(qt.entropy_vn(rho_A))
        scarEntangle.append(temp)

    scarEntangle = np.array(scarEntangle)
    plotScar = np.mean(scarEntangle, axis=0)

    plt.plot(tlist, plotScar)
    plt.title(f"Avged Thingamabob w/ {N} Qubits and {disorder} Disorder")
    plt.ylabel("Von Neumann Entropy")
    plt.xlabel("Time")
    plt.show()

def embed_scar_state_to_full(state, basisList, N):
    vec_constrained = state.full().flatten()
    vec_full = np.zeros(2**N, dtype=complex)

    for i, bitstr in enumerate(basisList):
        full_index = int(bitstr, 2)
        vec_full[full_index] = vec_constrained[i]

    return qt.Qobj(vec_full, dims=[[2]*N, [1]*N])